# BloomCast — high-quality model training (Kaggle)

Trains the production model on **real Tick Tick Bloom in-situ labels** and exports serving artifacts to `/kaggle/working/bloomcast_artifacts/`.

**Before running:**
1. Attach the competition CSVs (`train_labels.csv`, `metadata.csv` — download once from drivendata.org/competitions/143) as a Kaggle input dataset, **or** set `TICKTICKBLOOM_DIR` to their path.
2. Turn **internet ON** (the weather join pulls Open-Meteo archive data; cached to `/kaggle/working` so reruns are free).
3. Accelerator **GPU** recommended (torch CNN trains on it); CPU works, just slower.

Pipeline: labels → weather join → Optuna-tuned LightGBM → quantile models → GPU CNN → ensemble → calibration → export + verify.

In [ ]:
import os, sys
import numpy as np
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# Training-only deps (serving never needs these)
!pip install -q lightgbm scikit-learn pandas httpx scipy optuna shap

# BloomCast training library
if not os.path.isdir("/kaggle/working/BloomCast"):
    !git clone -q https://github.com/N-Garai/BloomCast.git /kaggle/working/BloomCast
sys.path.insert(0, "/kaggle/working/BloomCast/src/backend")
print("library ready")

In [ ]:
from pathlib import Path

# Point this at the attached dataset if the name differs
CANDIDATES = [os.environ.get("TICKTICKBLOOM_DIR", ""), "/kaggle/input/ticktickbloom"]
DATA_DIR = next((Path(p) for p in CANDIDATES if p and (Path(p) / "train_labels.csv").exists()), None)
assert DATA_DIR is not None, "train_labels.csv not found — attach the dataset or set TICKTICKBLOOM_DIR"
os.environ["TICKTICKBLOOM_DIR"] = str(DATA_DIR)
print("dataset:", DATA_DIR)

from ingestion.drivendata_loader import load_training_frame
rows = load_training_frame(DATA_DIR)
n_pos = sum(r["y"] for r in rows)
print(f"samples: {len(rows)} | positives: {n_pos} ({n_pos/len(rows):.1%}) | span: {rows[0]['date']} → {rows[-1]['date']}")

In [ ]:
from ml.training.real_labels import build_weather_frame

# Weather is cached beside the output so reruns never re-hit the API
CACHE = Path("/kaggle/working/weather_cache.csv")
frame = build_weather_frame(rows, cache_path=CACHE)
X, S, y, doy, wb_ids = frame["X"], frame["S"], frame["y"], frame["doy"], frame["wb_ids"]
print(f"X{X.shape} S{S.shape} positives={y.mean():.1%} skipped={frame['skipped']}")

In [ ]:
from ml.training.tuning import tune_lightgbm

N_TRIALS = int(os.environ.get("N_TRIALS", "30"))
tuned = tune_lightgbm(X, y, n_trials=N_TRIALS, n_splits=3)
print("best Brier:", round(tuned["best_brier"], 4))
print(tuned["params"])

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from ml.training.tuning import train_quantiles
from ml.training.lightgbm_branch import calibrate, ClimatologyBaseline, PersistenceBaseline, WeatherOnlyBaseline
from ml.training.scorecard import generate_scorecard
from ml.inference.predict import _proba, empirical_ci_half

P = tuned["params"]
tscv = TimeSeriesSplit(n_splits=5)
oof = np.zeros(len(y))
for tr, te in tscv.split(X):
    m = lgb.LGBMClassifier(**P)
    m.fit(X[tr].astype(np.float64), y[tr])
    oof[te] = m.predict_proba(X[te].astype(np.float64))[:, 1]
oof = __import__("numpy").where(__import__("numpy").isfinite(oof), __import__("numpy").nanmean(oof), oof)
final = lgb.LGBMClassifier(**P)
final.fit(X.astype(np.float64), y)
calibrator = calibrate(oof, y)
ci_half = empirical_ci_half(y, oof)
print(f"OOF AUC/Brier before calibration baseline — see scorecard below")

qmodels = train_quantiles(X, y, P)
print("quantiles trained: q05/q95")

from ml.training.lightgbm_branch import LightGBMBranch
branch = LightGBMBranch.__new__(LightGBMBranch)
branch.model = final

clim = ClimatologyBaseline().fit(X, y, doy, wb_ids)
persist = PersistenceBaseline().fit(X, y)
weather = WeatherOnlyBaseline().fit(X, y)
sc = generate_scorecard(y, oof, clim.predict(X, doy, wb_ids), persist.predict(X), weather.predict(X),
                        model_version="v2.2.0-kaggle-labels", sample_size=len(y), horizon_days=5)
print(f"OOF AUC={sc['auc']:.4f} Brier={sc['brier']:.4f} hit={sc['hit_rate']:.2%} fa={sc['false_alarm_rate']:.2%} ci_half={ci_half:.3f}")

In [ ]:
from ml.training.torch_cnn import train_torch_cnn, export_to_numpy
from ml.training.cnn_branch import BloomCNN
from ml.training.ensemble import Ensemble

CNN_EPOCHS = int(os.environ.get("CNN_EPOCHS", "15"))
tmodel, hist = train_torch_cnn(S, y, epochs=CNN_EPOCHS, lr=1e-3, batch_size=512)
print(f"CNN on {hist['device']} — loss {hist['loss'][0]:.4f} → {hist['loss'][-1]:.4f}")

np_weights = export_to_numpy(tmodel)
cnn = BloomCNN()
for k, v in np_weights.items():
    setattr(cnn, k, v)
# serving-exactness check: torch vs numpy embeddings must match
Xt = __import__("torch").as_tensor(S[:64])
_, emb_t = tmodel.forward(Xt)
assert np.allclose(emb_t.detach().cpu().numpy(), cnn.predict_embedding(S[:64]), atol=1e-5)
print("torch→numpy export verified exact")

emb_train = cnn.predict_embedding(S)
ens = Ensemble().fit(oof, emb_train, X, y)
print("ensemble fitted")

In [ ]:
import json
from datetime import datetime, timezone
from ml.training.artifacts import save_artifacts
from features.feature_store import FEATURE_NAMES
from ml.inference.predict import _profile_features, _shap_top, _band

OUT = Path("/kaggle/working/bloomcast_artifacts")
base = _profile_features()[3]
raw = float(final.predict_proba(base.reshape(1, -1).astype(np.float64))[0, 1])
p_head = float(calibrator.transform([raw])[0])
lo_head, hi_head = _band(p_head, base, qmodels, ci_half)
meta = {
    "model_version": "v2.2.0-kaggle-labels",
    "training_source": "tick-tick-bloom",
    "training_note": f"Kaggle-trained on {len(y)} Tick Tick Bloom samples; Optuna-tuned LightGBM + GPU CNN + quantiles.",
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "ci_half": ci_half,
    "oof_auc": sc["auc"],
    "oof_brier": sc["brier"],
    "optuna_params": P,
    "scorecard": json.loads(json.dumps(sc, default=float)),
    "headline": {
        "p_bloom": p_head,
        "ci_lo": lo_head,
        "ci_hi": hi_head,
        "shap_top_features": _shap_top(branch, base),
        "baseline_climatology": float(clim.predict(base.reshape(1, -1), doy[:1], wb_ids[:1])[0]),
    },
}
save_artifacts(OUT, lgbm_branch=branch, calibrator=calibrator, ensemble=ens,
               cnn=cnn, feature_names=FEATURE_NAMES, meta=meta,
               quantiles=qmodels)
print("exported:", sorted(p.name for p in OUT.iterdir()))

# Reload verification — the backend will do exactly this
from ml.training.artifacts import load_artifacts
re = load_artifacts(OUT, FEATURE_NAMES)
assert re is not None and re["meta"]["training_source"] == "tick-tick-bloom"
assert re["meta"].get("has_quantiles") is True
print("reload OK — download /kaggle/working/bloomcast_artifacts and commit it to src/backend/ml/artifacts/")

## After the run

1. Download **`bloomcast_artifacts/`** from `/kaggle/working` (Kaggle → Output → download).
2. Copy the 7 files into **`src/backend/ml/artifacts/`** in the repo and commit.
3. CI enforces the honesty rule (`test_committed_artifacts_are_real`) — only `tick-tick-bloom` provenance is accepted.
4. The next nightly seed + every deployment loads the model and runs **inference only**. No retraining, no labels, no weather join.